# RabindricGPT — Stage 1 Resumable Ingestion to 1.1 GB on Colab CPU
Streams FineWeb-2 Bengali dataset up to 1.1 GB target volume with:
1. Direct persistence to Google Drive (`/content/drive/MyDrive/RabindricGPT/data/raw/fineweb2/train.jsonl`).
2. Seamless resumption across Colab session disconnects.
3. Strict disjointness filtering and 4-work 5-gram leakage detection (*Shesher Kobita*, *Gitanjali*, *Jogajog*, *Chitrangada*).
4. Shard rebuilding and Drive `prepared/` packaging.

In [ ]:
# 1. Mount Google Drive
from pathlib import Path
import os
import sys
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive mount note: {e}")

# 2. Clone or sync RabindricGPT repository
repo_path = Path('/content/RabindricGPT')
if not repo_path.exists():
    !git clone https://github.com/ASMITSARKAR/RabindricGPT.git /content/RabindricGPT
else:
    !git -C /content/RabindricGPT pull --ff-only

os.chdir('/content/RabindricGPT')
if '/content/RabindricGPT' not in sys.path:
    sys.path.insert(0, '/content/RabindricGPT')

# 3. Install required dependencies
!pip install -q datasets pyarrow fsspec sentencepiece pyyaml

# 4. Link data/raw/fineweb2 to Google Drive for persistent resumption across disconnects
drive_base = Path('/content/drive/MyDrive/RabindricGPT')
drive_raw_fw2 = drive_base / 'data' / 'raw' / 'fineweb2'
drive_raw_fw2.mkdir(parents=True, exist_ok=True)

local_raw_fw2 = Path('/content/RabindricGPT/data/raw/fineweb2')
local_raw_fw2.parent.mkdir(parents=True, exist_ok=True)

drive_train = drive_raw_fw2 / 'train.jsonl'
if local_raw_fw2.exists() and not local_raw_fw2.is_symlink():
    local_train = local_raw_fw2 / 'train.jsonl'
    if local_train.exists() and not drive_train.exists():
        print(f"Seeding Drive with repo local train.jsonl ({local_train.stat().st_size:,} bytes)...")
        shutil.copy2(local_train, drive_train)
    shutil.rmtree(local_raw_fw2)

if not local_raw_fw2.exists():
    os.symlink(str(drive_raw_fw2), str(local_raw_fw2))
    print(f"Symlinked {local_raw_fw2} -> {drive_raw_fw2}")

# 5. Run resumable ingest stream towards 1.1 GB (1,100,000,000 bytes)
!python scripts/02_ingest_fineweb2.py --target-bytes 1100000000 --egen-chars 100000 --output-dir "{drive_raw_fw2}"

# 6. Clean and split raw data once target volume is reached
!python scripts/03_clean_and_split.py

# 7. Build shards across all 5 tokenizers
!python scripts/07_build_shards.py

# 8. Package prepared/ directory and sync archive + hashes to Drive
!python scripts/audits/build_prepared_drive.py

drive_prep = drive_base / 'prepared'
drive_prep.mkdir(parents=True, exist_ok=True)
!cp -r prepared/* /content/drive/MyDrive/RabindricGPT/prepared/
print('Finished! Shards, tokenizers, and hashes synced to Drive.')